In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

data = {
    "Size_sqft": [800, 900, 1000, 1100, 1200, 1400, 1600, 1800, 2000, 2200, 2400, 2600, 2800, 3000],
    "Bedrooms": [2,2,2,3,3,3,3,4,4,4,4,5,5,5],
    "Bathrooms": [1,1,2,2,2,2,3,3,3,3,4,4,4,4],
    "Age": [20,18,15,12,10,8,7,6,5,4,3,2,2,1],
    "Price": [150000,160000,180000,200000,210000,240000,260000,280000,300000,320000,350000,380000,410000,450000]
}

df = pd.DataFrame(data)

X = df.drop("Price", axis=1)
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

lr = LinearRegression()
ridge = Ridge()
rf = RandomForestRegressor(random_state=42)

lr.fit(X_train, y_train)
ridge.fit(X_train, y_train)
rf.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
ridge_pred = ridge.predict(X_test)
rf_pred = rf.predict(X_test)

def evaluate(name, y_true, y_pred):
    print(name)
    print("MSE:", mean_squared_error(y_true, y_pred))
    print("MAE:", mean_absolute_error(y_true, y_pred))
    print("R2 Score:", r2_score(y_true, y_pred))
    print()

evaluate("Linear Regression", y_test, lr_pred)
evaluate("Ridge Regression", y_test, ridge_pred)
evaluate("Random Forest", y_test, rf_pred)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10],
    "min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_

best_pred = best_rf.predict(X_test)

print("Tuned Random Forest Performance")
evaluate("Best Random Forest", y_test, best_pred)

def predict_house_price(size, bedrooms, bathrooms, age):
    new_data = pd.DataFrame([[size, bedrooms, bathrooms, age]],
                            columns=["Size_sqft", "Bedrooms", "Bathrooms", "Age"])
    new_scaled = scaler.transform(new_data)
    prediction = best_rf.predict(new_scaled)
    return prediction[0]

def get_float(prompt):
    while True:
        value = input(prompt)
        if value.strip() == "":
            print("Please enter a value.")
            continue
        try:
            return float(value)
        except ValueError:
            print("Invalid number. Try again.")

def get_int(prompt):
    while True:
        value = input(prompt)
        if value.strip() == "":
            print("Please enter a value.")
            continue
        try:
            return int(value)
        except ValueError:
            print("Invalid number. Try again.")

size = get_float("Enter house size: ")
bedrooms = get_int("Enter bedrooms: ")
bathrooms = get_int("Enter bathrooms: ")
age = get_int("Enter house age: ")

price = predict_house_price(size, bedrooms, bathrooms, age)

print("Predicted House Price:", price)

Linear Regression
MSE: 127924856.01669829
MAE: 8253.829860598073
R2 Score: 0.9865184577968351

Ridge Regression
MSE: 101153200.53191425
MAE: 9906.162122530097
R2 Score: 0.9893398266418357

Random Forest
MSE: 278700000.0
MAE: 15600.0
R2 Score: 0.9706288056206089

Tuned Random Forest Performance
Best Random Forest
MSE: 278700000.0
MAE: 15600.0
R2 Score: 0.9706288056206089



In [2]:
import pandas as pd
import seaborn as sns   # <-- Add this line

df = sns.load_dataset("tips")
print(df.head())

   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3
3       23.68  3.31    Male     No  Sun  Dinner     2
4       24.59  3.61  Female     No  Sun  Dinner     4


In [3]:
df_encoded = pd.get_dummies(df, columns=["sex", "smoker", "day", "time"], drop_first=True)

print(df_encoded.head())

   total_bill   tip  size  sex_Female  smoker_No  day_Fri  day_Sat  day_Sun  \
0       16.99  1.01     2        True       True    False    False     True   
1       10.34  1.66     3       False       True    False    False     True   
2       21.01  3.50     3       False       True    False    False     True   
3       23.68  3.31     2       False       True    False    False     True   
4       24.59  3.61     4        True       True    False    False     True   

   time_Dinner  
0         True  
1         True  
2         True  
3         True  
4         True  


In [4]:
import pandas as pd
import seaborn as sns

# Load dataset
df = sns.load_dataset("tips")

# Select only categorical columns
categorical_df = df.select_dtypes(include=["object", "category"])

print(categorical_df.head())

      sex smoker  day    time
0  Female     No  Sun  Dinner
1    Male     No  Sun  Dinner
2    Male     No  Sun  Dinner
3    Male     No  Sun  Dinner
4  Female     No  Sun  Dinner


In [5]:
day_encoded = pd.get_dummies(df["day"])
print(day_encoded.head())

    Thur    Fri    Sat   Sun
0  False  False  False  True
1  False  False  False  True
2  False  False  False  True
3  False  False  False  True
4  False  False  False  True


In [6]:
import pandas as pd
import seaborn as sns

# Load dataset
df = sns.load_dataset("tips")

# Separate numeric and categorical columns
numeric_df = df.select_dtypes(include=["number"])
categorical_df = df.select_dtypes(include=["object", "category"])

# Apply one-hot encoding to categorical columns
categorical_encoded = pd.get_dummies(categorical_df)

# Join numeric columns with encoded columns
final_df = pd.concat([numeric_df, categorical_encoded], axis=1)

print(final_df.head())

   total_bill   tip  size  sex_Male  sex_Female  smoker_Yes  smoker_No  \
0       16.99  1.01     2     False        True       False       True   
1       10.34  1.66     3      True       False       False       True   
2       21.01  3.50     3      True       False       False       True   
3       23.68  3.31     2      True       False       False       True   
4       24.59  3.61     4     False        True       False       True   

   day_Thur  day_Fri  day_Sat  day_Sun  time_Lunch  time_Dinner  
0     False    False    False     True       False         True  
1     False    False    False     True       False         True  
2     False    False    False     True       False         True  
3     False    False    False     True       False         True  
4     False    False    False     True       False         True  


In [7]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

# Load dataset
df = sns.load_dataset("tips")

# Convert categorical columns to numeric using one-hot encoding
df = pd.get_dummies(df, drop_first=True)

# Divide data into features (X) and label (y)
X = df.drop("tip", axis=1)   # Features
y = df["tip"]                # Target variable

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Display shapes of datasets
print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Training labels shape:", y_train.shape)
print("Testing labels shape:", y_test.shape)

Training features shape: (195, 8)
Testing features shape: (49, 8)
Training labels shape: (195,)
Testing labels shape: (49,)


In [8]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
df = sns.load_dataset("tips")

# Convert categorical columns to numeric using one-hot encoding
df = pd.get_dummies(df, drop_first=True)

# Divide data into Features (X) and Label (y)
X = df.drop("tip", axis=1)
y = df["tip"]

# Split dataset into Training and Test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Data Scaling / Normalization
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Display dataset shapes
print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (195, 8) (195,)
Test set: (49, 8) (49,)


In [9]:
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
df = sns.load_dataset("tips")

# Convert categorical columns into numeric (One-Hot Encoding)
df = pd.get_dummies(df, drop_first=True)

# Separate Features (X) and Target (y)
X = df.drop("tip", axis=1)
y = df["tip"]

# Split data into Training and Test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Importing Linear Regression model from sklearn
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

Mean Absolute Error: 0.6671331480264893
Mean Squared Error: 0.7033566017436106
R2 Score: 0.43730181943482493
